In [1]:
%pip install -q transformers datasets accelerate scikit-learn pandas

In [2]:
import pandas as pd

data = {
    "code": [
        "password = 'admin123'",
        "api_key = 'my_secret_key'",
        "query = 'SELECT * FROM users WHERE id = ' + user_id",
        "eval(user_input)",
        "def divide(a, b): return a / b",
        "items = [1, 2, 3]\nprint(items[10])",
        "result = 10 / number",
        "value = int('hello')",
        "def add(a,b):\n return a+b",
        "x=1",
        "def calculate_total(price, tax):\n    return price + tax",
        "def greet(name):\n    return 'Hello ' + name",
        "def multiply(a, b):\n    return a * b",
        "def square(n):\n    return n * n",
        "def get_name(user):\n    return user.name",
        "for i in range(5):\n    print(i)",
    ],
    "label": [
        "security",
        "security",
        "security",
        "security",
        "bug",
        "bug",
        "bug",
        "bug",
        "style",
        "style",
        "no_issue",
        "no_issue",
        "no_issue",
        "no_issue",
        "no_issue",
        "no_issue",
    ]
}

df = pd.DataFrame(data)

print(df.head())
print("\nDataset size:", len(df))
print("\nLabel distribution:")
print(df["label"].value_counts())

                                                code     label
0                              password = 'admin123'  security
1                          api_key = 'my_secret_key'  security
2  query = 'SELECT * FROM users WHERE id = ' + us...  security
3                                   eval(user_input)  security
4                     def divide(a, b): return a / b       bug

Dataset size: 16

Label distribution:
label
no_issue    6
security    4
bug         4
style       2
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Convert text labels into numerical IDs
label_encoder = LabelEncoder()

df["label_id"] = label_encoder.fit_transform(df["label"])

print("Label mapping:")

for label, label_id in zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
):
    print(f"{label} -> {label_id}")

# Split dataset into training and testing sets
train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=df["label_id"]
)

print("\nTraining examples:", len(train_df))
print("Testing examples:", len(test_df))

Label mapping:
bug -> 0
no_issue -> 1
security -> 2
style -> 3

Training examples: 12
Testing examples: 4


In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "distilbert/distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the labels
id2label = {
    0: "bug",
    1: "no_issue",
    2: "security",
    3: "style"
}

label2id = {
    label: idx
    for idx, label in id2label.items()
}

# Load pretrained model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

print("Pretrained model loaded successfully!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Pretrained model loaded successfully!


In [5]:
from datasets import Dataset

# Keep only the columns needed for training
train_data = Dataset.from_pandas(
    train_df[["code", "label_id"]].reset_index(drop=True)
)

test_data = Dataset.from_pandas(
    test_df[["code", "label_id"]].reset_index(drop=True)
)

# Rename label_id to label
train_data = train_data.rename_column("label_id", "label")
test_data = test_data.rename_column("label_id", "label")


# Convert Python code into tokens
def tokenize_function(examples):
    return tokenizer(
        examples["code"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


tokenized_train = train_data.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_data.map(
    tokenize_function,
    batched=True
)

# Remove original code text from model inputs
tokenized_train = tokenized_train.remove_columns(["code"])
tokenized_test = tokenized_test.remove_columns(["code"])

print("Training data prepared!")
print(tokenized_train[0].keys())

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Training data prepared!
dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [6]:
import torch
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# Select GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

# Configure training
training_args = TrainingArguments(
    output_dir="./code_review_results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=42
)

# Dynamically pad inputs within each batch
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Training configuration ready!")

Training device: cpu
Training configuration ready!


In [7]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = logits.argmax(axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Start fine-tuning!
train_result = trainer.train()

print("\nFine-tuning completed!")
print(train_result.metrics)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.337855,0.500000,0.222222
2,No log,1.317988,0.500000,0.222222
3,No log,1.299720,0.500000,0.222222
4,No log,1.289479,0.500000,0.222222
5,No log,1.285404,0.500000,0.222222


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning completed!
{'train_runtime': 136.3229, 'train_samples_per_second': 0.44, 'train_steps_per_second': 0.11, 'total_flos': 1987081850880.0, 'train_loss': 1.3153592427571614, 'epoch': 5.0}


In [8]:
results = trainer.evaluate()

print("Evaluation results:")

for key, value in results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
No log,1.285404,5,0.500000,0.222222


Evaluation results:
eval_loss: 1.2854044437408447
eval_accuracy: 0.5
eval_f1_macro: 0.2222222222222222


In [9]:
import torch

def predict_code_category(code):
    model.eval()

    inputs = tokenizer(
        code,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_id = probabilities.argmax(dim=-1).item()

    predicted_label = model.config.id2label[predicted_id]

    confidence = probabilities[0, predicted_id].item()

    return {
        "code": code,
        "predicted_category": predicted_label,
        "confidence": round(confidence, 4)
    }


# Test with new examples
examples = [
    "password = 'secret123'",
    "def multiply(a, b):\n    return a * b",
    "x=1"
]

for example in examples:
    print(predict_code_category(example))

{'code': "password = 'secret123'", 'predicted_category': 'no_issue', 'confidence': 0.2762}
{'code': 'def multiply(a, b):\n    return a * b', 'predicted_category': 'no_issue', 'confidence': 0.3053}
{'code': 'x=1', 'predicted_category': 'no_issue', 'confidence': 0.283}


In [10]:
save_path = "./fine_tuned_code_review_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Fine-tuned model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully!
